# Notebook 05 — Isolation Forest Anomaly Detection

**Input:** `data/processed/features.csv` (raw features) and `data/processed/predictions_so_far.csv` (from Notebook 04)

**Goal:** Apply tree-ensemble-based anomaly detection (Isolation Forest) directly on the full 42-dimensional feature matrix, compare design tradeoffs with clustering methods, and combine predictions for final evaluation in Notebook 06.

> **Strict Label Isolation:** Ground-truth labels (`is_anomaly`, `anomaly_type`) are stored separately and **NEVER** passed to the model or used for hyperparameter tuning.


In [ ]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Prevent OpenMP / BLAS / Joblib thread oversubscription and process contention on Windows
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
os.environ.setdefault('LOKY_MAX_CPU_COUNT', '4')

sys.path.insert(0, os.path.join('..', 'src'))
from detection_models import IsolationForestDetector

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

# Load features and previous predictions
df = pd.read_csv('../data/processed/features.csv', parse_dates=['timestamp'])
preds_df = pd.read_csv('../data/processed/predictions_so_far.csv', parse_dates=['timestamp'])

ID_COLS    = ['timestamp', 'server_id', 'service_type']
LABEL_COLS = ['is_anomaly', 'anomaly_type']
FEAT_COLS  = [c for c in df.columns if c not in ID_COLS + LABEL_COLS]

X = df[FEAT_COLS].values
assert all(c not in FEAT_COLS for c in LABEL_COLS), 'LABEL LEAK!'
print(f'Feature matrix X (raw features): {X.shape}')


## 1. Why Isolation Forest on Raw Features (No PCA)?

### Architectural Rationale & Tradeoffs:
1. **Clustering Methods (K-Means & DBSCAN - Notebook 04):** Rely on Euclidean distance metrics ($L_2$ norm). In 42-dimensional space, Euclidean distance suffers from the *Curse of Dimensionality* — distances between points homogenize, causing density-based boundaries to blur. Therefore, PCA dimensionality reduction (to ~17 components) is a mandatory preprocessing step for clustering models.
2. **Tree Ensembles (Isolation Forest - Notebook 05):** Isolates anomalies by randomly selecting a feature and randomly splitting between its minimum and maximum values via recursive binary partitioning (iTrees). Anomalies, being few and quantitatively distinct, require far fewer random splits (`path length $h(x)$`) to isolate than normal operating points.
3. **No PCA Needed:** Because axis-aligned random partitioning works natively across individual feature distributions without computing multi-dimensional Euclidean distances, Isolation Forest thrives directly on the raw 42-dimensional feature matrix `X`. This preserves localized single-metric anomalies (e.g., sudden `cpu_usage` spikes) that might be smoothed or projected away during PCA variance reduction.


In [ ]:
# Load pre-trained PCA coordinates for visual comparison plots only
X_pca = np.load('../data/processed/X_pca.npy')
print(f'Loaded X_pca ({X_pca.shape}) for 2D visualization.')


## 2. Fit Isolation Forest & Score Distribution

We set `contamination = 0.05` as our tunable hyperparameter, representing an approximate domain estimate that ~5% of server logs reflect anomalous behavior. **We do NOT tune this parameter against ground-truth labels.**


In [ ]:
CONTAMINATION = 0.05
print(f'Fitting Isolation Forest (n_estimators=200, contamination={CONTAMINATION})...')

if_detector = IsolationForestDetector(contamination=CONTAMINATION, random_state=42)
if_detector.fit(X)

# Compute continuous anomaly scores and binary anomaly predictions
if_scores = if_detector.score(X)     # Higher = more anomalous (-decision_function)
if_preds  = if_detector.predict(X)   # 1 = anomaly, 0 = normal

print(f'Flagged as anomaly: {if_preds.sum():,} ({if_preds.mean()*100:.1f}%)')
preds_df['iforest_score']   = if_scores
preds_df['iforest_anomaly'] = if_preds


In [ ]:
# Anomaly Score Distribution Histogram
fig, ax = plt.subplots(figsize=(10, 4.5))
sns.histplot(if_scores, bins=80, kde=True, color='purple', ax=ax)

# Find threshold score corresponding to top 5% contamination
threshold_score = np.percentile(if_scores, 100 * (1 - CONTAMINATION))
ax.axvline(threshold_score, color='crimson', ls='--', lw=2, label=f'Contamination Cut-off ({threshold_score:.4f})')

ax.set_xlabel('Isolation Forest Anomaly Score (-decision_function)')
ax.set_ylabel('Log Count')
ax.set_title('Isolation Forest Anomaly Score Distribution', fontsize=12, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('../results/iforest_score_dist.png', dpi=110, bbox_inches='tight')
plt.show()


## 3. 2D Visualization (PCA Scatter vs Raw Metric Scatter)

Even though Isolation Forest operated on raw 42D features, visualizing the flags across the first two principal components (`PC1 vs PC2`) and raw key infrastructure metrics (`cpu_usage vs response_time_ms`) illustrates how tree partitioning isolates anomalous regions compared to distance clustering.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Isolation Forest Anomaly Flags — 2D Projections', fontsize=13, fontweight='bold')

# Left: PC1 vs PC2 scatter
axes[0].scatter(X_pca[if_preds==0, 0], X_pca[if_preds==0, 1],
                c='lightsteelblue', s=2, alpha=0.3, label='Normal (0)')
axes[0].scatter(X_pca[if_preds==1, 0], X_pca[if_preds==1, 1],
                c='darkviolet', s=10, alpha=0.7, label='Flagged Anomaly (1)')
axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2')
axes[0].set_title('Flags Projected onto PCA Space (PC1 vs PC2)')
axes[0].legend(fontsize=9, markerscale=3)

# Right: Raw CPU vs Response Time scatter
axes[1].scatter(df.loc[if_preds==0, 'cpu_utilization_pct'], df.loc[if_preds==0, 'response_time_ms'],
                c='lightsteelblue', s=2, alpha=0.3, label='Normal (0)')
axes[1].scatter(df.loc[if_preds==1, 'cpu_utilization_pct'], df.loc[if_preds==1, 'response_time_ms'],
                c='darkviolet', s=10, alpha=0.7, label='Flagged Anomaly (1)')
axes[1].set_xlabel('CPU Utilization (%)'); axes[1].set_ylabel('Response Time (ms)')
axes[1].set_title('Flags in Raw Metric Space (CPU vs Response Time)')
axes[1].legend(fontsize=9, markerscale=3)

plt.tight_layout()
plt.savefig('../results/iforest_scatter.png', dpi=110, bbox_inches='tight')
plt.show()


## 4. Save Combined Predictions for Evaluation

We combine all three unsupervised model predictions (`kmeans_anomaly`, `dbscan_anomaly`, `iforest_anomaly`) into a single master predictions dataset for rigorous ground-truth evaluation in **Notebook 06**.


In [ ]:
preds_df.to_csv('../data/processed/all_predictions.csv', index=False)
print('Saved master predictions to: data/processed/all_predictions.csv')
print(f'Columns ({len(preds_df.columns)}): {preds_df.columns.tolist()}')
print('\nNext Step: Run Notebook 06 (06_evaluation_and_dashboard_prep.ipynb)')
